In [1]:
# https://docs.pytorch.org/tutorials/intermediate/ddp_tutorial.html#combining-ddp-with-model-parallelism

In [2]:
# Combining DP with MP

In [3]:
import os
import sys
import tempfile
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.optim as optim
import torch.multiprocessing as mp

from torch.nn.parallel import DistributedDataParallel as DDP

In [4]:
class ToyMpModel(nn.Module):
  def __init__(self, dev0, dev1):
    super().__init__()

    self.dev0 = dev0
    self.dev1 = dev1

    self.layer1 = nn.Linear(10, 20).to(self.dev0)
    self.relu = nn.ReLU()
    self.layer2 = nn.Linear(20, 5).to(self.dev1)

  def forward(self, x):
    x = x.to(self.dev0)
    out1 = self.layer1(x)
    out1 = self.relu(out1)
    out1 = out1.to(self.dev1)
    out2 = self.layer2(out1)
    return out2

In [5]:
def setup(rank, world_size):
  os.environ["MASTER_ADDR"] = "localhost"
  os.environ["MASTER_PORT"] = "8080"

  dist.init_process_group(backend="nccl", rank=rank, world_size=world_size)

def cleanup():
  dist.destroy_process_group()

In [6]:
def demo_model_parallel(rank, world_size):
  setup(rank, world_size)

  dev0 = 0
  dev1 = 0

  mp_model = ToyMpModel(dev0, dev1)
  ddp_model = DDP(mp_model)
  loss_fn = nn.MSELoss()
  optimizer = optim.SGD(ddp_model.parameters(), lr=0.001)

  output = ddp_model(torch.randn(20, 10))

  optimizer.zero_grad()

  labels = torch.randn(20, 5).to(dev1)  # Put this on the second device as output is ont he second d
  loss = loss_fn(output, labels)
  loss.backward()

  optimizer.step()

  cleanup()